# Clean RL Evaluation Notebook

This notebook properly evaluates all agents by:
1. Running episodes and collecting cumulative metrics
2. Using reward as the primary metric (combines latency + energy)
3. Tracking cumulative energy per episode (not per-task)
4. Comparing agent strategies


In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# NumPy compatibility
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

print('✓ Setup complete')

In [ ]:
# Import gym and veins
import gym
import veins_gym

# Helper to safely close environment
def close_env_safe():
    global env
    try:
        if 'env' in globals():
            env.close()
            print('✓ Closed environment')
    except Exception as e:
        pass

print('✓ Imports ready')

In [ ]:
# Register and create environment
gym.register(
    id='veins-straight-v1',
    entry_point='veins_gym:VeinsEnv',
    kwargs={
        'scenario_dir': 'scenario',
        'timeout': 30.0,
        'print_veins_stdout': False,
        'user_interface': 'Cmdenv',
        'config': 'StraightRoad',
        'run_veins': True,
    },
)

env = gym.make('veins-straight-v1')
print('✓ Environment created')
print(f'  Action space: {env.action_space}')
print(f'  Observation space: {env.observation_space}')

In [ ]:
# Test environment
def _extract_obs(result):
    if isinstance(result, tuple):
        return result[0]
    return result

print('Testing environment with 3 random actions...')
obs = _extract_obs(env.reset())

for step in range(3):
    action = env.action_space.sample()
    result = env.step(action)
    if len(result) == 5:
        obs, reward, terminated, truncated, info = result
        done = terminated or truncated
    else:
        obs, reward, done, info = result
    
    print(f'Step {step}: action={action}, reward={float(reward):.4f}, done={done}')

close_env_safe()

In [ ]:
# Load trained RL model
import torch
import torch.nn as nn

class Policy(nn.Module):
    def __init__(self, obs_dim=14, n_actions=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, n_actions),
        )

    def forward(self, x):
        return self.net(x)

def preprocess_obs(obs):
    x = np.asarray(obs, dtype=np.float32).copy()
    x[0] /= 30.0
    x[1:4] /= 1500.0
    x[4] /= 20.0
    x[8:11] /= 50.0
    return x

# Load model
model_path = Path('./models/bandit_policy.pt')
if model_path.exists():
    checkpoint = torch.load(model_path, map_location='cpu')
    policy = Policy()
    policy.load_state_dict(checkpoint['model_state'])
    policy.eval()
    print('✓ Loaded trained bandit policy')
else:
    policy = None
    print('⚠ Model file not found')

In [ ]:
# Define agent policies
def policy_local(obs, env):
    """Always process locally"""
    return 0

def policy_random(obs, env):
    """Random action"""
    return env.action_space.sample()

def policy_greedy(obs, env):
    """Greedy: offload to nearest free RSU, fallback to local"""
    from nearest_free_rsu_agent import pick_action
    return int(pick_action(obs))

def policy_bandit(obs, env):
    """RL-trained policy"""
    if policy is None:
        return 0
    x = preprocess_obs(obs)
    x_t = torch.tensor(x, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        logits = policy(x_t)
    return int(logits.argmax(dim=-1).item())

print('✓ Policies defined')

In [ ]:
# Evaluate agents
print('="' * 40)
print('EVALUATING AGENTS')
print('="' * 40)

agents = [
    ('Local', policy_local),
    ('Random', policy_random),
    ('Greedy', policy_greedy),
]

if policy is not None:
    agents.append(('Bandit RL', policy_bandit))

n_episodes = 3
all_results = []

for agent_name, policy_fn in agents:
    print(f'\n{agent_name}:')
    
    episode_rewards = []
    episode_offload_ratios = []
    
    for ep_num in range(n_episodes):
        close_env_safe()
        obs = _extract_obs(env.reset())
        done = False
        
        total_reward = 0.0
        actions = []
        step_count = 0
        max_steps = 50
        
        while not done and step_count < max_steps:
            action = policy_fn(obs, env)
            actions.append(action)
            
            result = env.step(action)
            if len(result) == 5:
                obs, reward, terminated, truncated, info = result
                done = terminated or truncated
            else:
                obs, reward, done, info = result
            
            total_reward += float(reward)
            step_count += 1
        
        offload_ratio = np.mean([a > 0 for a in actions]) if actions else 0
        episode_rewards.append(total_reward)
        episode_offload_ratios.append(offload_ratio)
        
        print(f'  Ep {ep_num+1}: reward={total_reward:.4f}, offload={offload_ratio:.1%}, steps={step_count}')
    
    result_dict = {
        'agent': agent_name,
        'mean_reward': np.mean(episode_rewards),
        'std_reward': np.std(episode_rewards),
        'mean_offload': np.mean(episode_offload_ratios),
    }
    all_results.append(result_dict)
    
    print(f'  Summary: reward={result_dict["mean_reward"]:.4f} ± {result_dict["std_reward"]:.4f}')

close_env_safe()

df = pd.DataFrame(all_results)
print(f'\n{"="*40}')
print('RESULTS')
print(f'{"="*40}')
print(df.to_string(index=False))

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Reward comparison
axes[0].bar(df['agent'], df['mean_reward'], 
            yerr=df['std_reward'], capsize=5, alpha=0.7,
            color=['blue', 'orange', 'green', 'red'][:len(df)])
axes[0].set_ylabel('Total Reward per Episode')
axes[0].set_title('Reward: Latency + Energy Balance')
axes[0].grid(True, alpha=0.3)

# Offload strategy
axes[1].bar(df['agent'], df['mean_offload'], alpha=0.7,
            color=['blue', 'orange', 'green', 'red'][:len(df)])
axes[1].set_ylabel('Offload Ratio')
axes[1].set_title('Offloading Strategy')
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nInterpretation:')
print('- Higher reward = better balanced decisions')
print('- Local: Slow but efficient (low offload ratio)')
print('- Greedy: Fast but expensive (high offload ratio)')
print('- RL: Should balance both')